# Warpmate, Step 6c: the frontier GPU solver

The original GPU solver checks **every** unsolved position in every round. The new **frontier** solver only checks positions that can actually change: when a position is solved, it walks backward (un-moves) to the positions that lead to it and marks only those for the next round.

This notebook:

1. Builds Warpmate and runs the tests (both GPU algorithms must match the CPU byte for byte).
2. Solves all five 4-piece endgames with the frontier solver.
3. Benchmarks **frontier vs full scan** on the GPU for every 4-piece endgame, checking the tables are identical.
4. Re-measures KRvK with the current solver (the old KRvK numbers came from code that has been removed).
5. Times the CPU on KQvKR with the frontier algorithm, and profiles the GPU with Nsight Systems.
6. Optionally re-checks the new tables against Syzygy.

**Before you start:** *Runtime > Change runtime type > T4 GPU > Save*, then run the cells top to bottom.

In [ ]:
!nvidia-smi

## Get the code

If `warpmate-colab.zip` is in the **Files** panel on the left (drag it there), it's used; otherwise the code is cloned from GitHub.

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/warpmate-colab.zip'):
    !mkdir -p Warpmate && unzip -o -q warpmate-colab.zip -d Warpmate
    print('Using the uploaded zip')
elif os.path.isdir('/content/Warpmate/.git'):
    # Already cloned in this session: update it instead of starting over, so a
    # re-run keeps earlier results (tables, dataset, trained model).
    !cd /content/Warpmate && git pull -q
    print('Updated the existing copy from GitHub')
else:
    !rm -rf Warpmate && git clone -q https://github.com/vasishtk1/Warpmate.git
    print('Cloned from GitHub')
%cd /content/Warpmate
!test -f src/egtb/unmove.h && echo "Frontier code present" || echo "WARNING: this copy doesn't have the frontier code yet"

## Build

Look for `Warpmate GPU support: ON`.

In [ ]:
import os
if os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -4
!cmake --build build -j 2>&1 | tail -6

## Tests

- `egtb`: un-moves are the exact reverse of moves, and the frontier CPU solver matches the full scan.
- `egtb-gpu`: both GPU algorithms (full scan and frontier) give tables identical to the CPU for every 3-piece endgame, solve the same positions in every round, and the GPU frontier checks exactly the same positions as the CPU frontier.

In [ ]:
!ctest --test-dir build --output-on-failure

## Solve all 4-piece endgames with the frontier solver

`warpmate-gen --gpu` now uses the frontier algorithm. The longest mates should be the same as in step 6b: KQvKR 35, KRvKR 19, KRvKB 29, KRvKN 40, KRvKP 43.

In [ ]:
!./build/warpmate-gen --gpu KRvKP 2>&1 | tee gen_gpu.log

## GPU: frontier vs full scan, every 4-piece endgame

For each endgame: 1 warm-up, then 3 timed runs of each algorithm. Every table must be identical. KQvKR also prints the per-round counts: how many positions each algorithm checked, and how many were solved. About 2 minutes in total.

In [ ]:
!./build/warpmate-bench KQvKR --cpu-trials 0 --gpu-trials 3 --rounds
for t in ['KRvKR', 'KRvKB', 'KRvKN', 'KRvKP']:
    print('\n' + '=' * 70)
    !./build/warpmate-bench {t} --cpu-trials 0 --gpu-trials 3

## Re-measure KRvK

The KRvK numbers in the docs came from the first, KRvK-only solvers, which have been removed. These runs measure the current solver: the CPU with both algorithms (1 and 2 threads) and both GPU algorithms. Takes under a minute.

In [ ]:
!echo "CPU: $(lscpu | grep 'Model name' | sed 's/Model name: *//') ($(nproc) threads)"
!./build/warpmate-bench KRvK --cpu-trials 5 --gpu-trials 5 --threads 1 --cpu-algo full
!./build/warpmate-bench KRvK --cpu-trials 5 --gpu-trials 1 --threads 2 --cpu-algo full | tail -9
!./build/warpmate-bench KRvK --cpu-trials 5 --gpu-trials 1 --threads 1 | tail -9
!./build/warpmate-bench KRvK --cpu-trials 5 --gpu-trials 1 --threads 2 | tail -9

## CPU vs GPU on KQvKR (frontier on both)

The CPU now uses the frontier algorithm too, so this compares the best CPU version against the best GPU version. In step 6b the full-scan CPU took about 10 minutes; this should be quicker, but it may still take a few minutes.

In [ ]:
!./build/warpmate-bench KQvKR --cpu-trials 1 --gpu-trials 3

## Profile with Nsight Systems

Shows how GPU time splits between the kernels (`mark_parents_kernel`, `frontier_step_kernel`, ...) and memory operations when solving KQvKR with the frontier solver. Skips itself if `nsys` isn't installed.

In [ ]:
!if command -v nsys >/dev/null; then nsys profile --stats=true --force-overwrite true -o frontier_profile ./build/warpmate-gen --gpu -d /tmp/profile_tables KQvKR 2>&1 | tail -60; else echo "nsys not installed here, skipping"; fi

## Optional: re-check against Syzygy

The benchmark above already proves the frontier tables are identical to the full-scan ones, which passed Syzygy in step 6b. This re-checks the new tables directly (a random sample of 1,000,000 rows per 4-piece table, about 8 minutes).

In [ ]:
!pip -q install chess
!mkdir -p syzygy
!for t in KQvK KRvK KBvK KNvK KPvK KQvKR KRvKR KRvKB KRvKN KRvKP; do wget -q -nc -P syzygy https://tablebase.lichess.ovh/tables/standard/3-4-5-wdl/$t.rtbw; done
!python scripts/verify_syzygy.py --syzygy syzygy --sample 1000000 tables/KQvKR.wmtb tables/KRvKR.wmtb tables/KRvKB.wmtb tables/KRvKN.wmtb tables/KRvKP.wmtb

## Done

Copy the output of every cell from **Tests** onward and paste it back.